# Model Benchmark

We compare eight classifier families/configurations with the same preprocessing, stratified split, and metrics aligned with the maintenance problem.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import recall_score

PROJECT_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing src/ and data/.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_pump_it_up_data
from src.modeling import build_classifier_pipeline, classification_metrics, portfolio_model_specs, stratified_split

train_values, train_labels, _ = load_pump_it_up_data(PROJECT_ROOT / 'data')
train = train_values.merge(train_labels, on='id', validate='one_to_one')
X = train.drop(columns=['status_group'])
y = train['status_group']
X_train, X_validation, y_train, y_validation = stratified_split(X, y)

In [2]:
rows = []
for name, estimator in portfolio_model_specs().items():
    pipeline = build_classifier_pipeline(
        X_train, estimator, scale_features=name.startswith('logistic')
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_validation)
    metrics = classification_metrics(y_validation, predictions)
    repair_recall = recall_score(
        y_validation, predictions, labels=['functional needs repair'], average=None
    )[0]
    rows.append({
        'model': name,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'repair_recall': repair_recall,
    })

comparison = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).round(4)
display(comparison)

,model,accuracy,macro_f1,repair_recall
2,random_forest_balanced,0.7989,0.6963,0.4299
3,extra_trees_balanced,0.8028,0.6829,0.3221
4,hist_gradient_boosting,0.7882,0.6465,0.2225
6,lightgbm_balanced,0.7119,0.6356,0.6941
5,xgboost,0.7755,0.6110,0.1553
1,decision_tree_balanced,0.6833,0.5974,0.5295
7,catboost_balanced,0.6672,0.5962,0.7045
0,logistic_regression_balanced,0.5229,0.4653,0.5261


## Reading the results

No single metric is sufficient: Extra Trees provides the highest accuracy, Random Forest the best macro F1, and CatBoost/LightGBM the highest minority-class recall. Final selection should depend on the operational cost of each error; for maintenance, recall for pumps that need repair deserves explicit consideration.